# TripleNeurons – Notebook 4: Chat Interface

**Project:** RAG information assistant for Philippine waste classification and segregation
**Group:** TripleNeurons, ITC-C508, José Rizal University

This notebook gives the chatbot a custom web interface. It does **not** rebuild anything: it loads the same Chroma index, configuration and `rag_core.py` saved by Notebook 2, so the answers come from exactly the pipeline that Notebook 3 evaluated (C4 chunks, all-MiniLM-L6-v2, Top-3 retrieval, Qwen2.5-1.5B-Instruct, greedy decoding).

What the interface adds over the plain Gradio ChatInterface of Notebook 2:
- a sidebar with the five knowledge-base documents and example questions;
- source cards for every answer (document, page, similarity) that open to show the retrieved passage, so the user can check the claim;
- a clearly marked "not in the documents" state when the chatbot refuses;
- a notice that the assistant is information only and that local schedules must be confirmed with the LGU;
- a usage log of every question asked (App_Usage sheet).

**How to run:** Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all. In Section 3, upload `TripleNeurons_NB2_Outputs.zip`.

## 1. Install packages

In [1]:
%pip -q install -U transformers accelerate sentence-transformers langchain-community langchain-text-splitters langchain-huggingface langchain-chroma chromadb gradio openpyxl

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
google-cloud-bigquery 3.47.0 requires python-dateut

## 2. Setup and logging helpers

In [2]:
import datetime, json, os, re, shutil, time, zipfile
from pathlib import Path
from zoneinfo import ZoneInfo

LOGS = {}


def now_ph():
    """Timestamp in Philippine time (Colab clocks run in UTC)."""
    return datetime.datetime.now(ZoneInfo("Asia/Manila")).isoformat(timespec="seconds")


def log(sheet, **row):
    """Add one timestamped row to a log sheet of Deliverable 4."""
    LOGS.setdefault(sheet, []).append(
        {"timestamp": now_ph(), "member": MEMBER, "notebook": NOTEBOOK_ID, **row})


def log_issue(stage, message):
    """Record a problem honestly instead of hiding it."""
    log("Issues_Log", stage=stage, issue=str(message))
    print(f"[ISSUE logged] {stage}: {message}")


def save_logs(folder):
    """Write every log sheet as <notebook>_<sheet>.csv so the next notebook can merge them."""
    import pandas as pd
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    for sheet, rows in LOGS.items():
        pd.DataFrame(rows).to_csv(folder / f"{NOTEBOOK_ID}__{sheet}.csv", index=False)
    print("Saved log sheets:", sorted(LOGS))


def zip_folder(folder, zip_path):
    folder = Path(folder)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in sorted(folder.rglob("*")):
            if p.is_file():
                zf.write(p, p.relative_to(folder))
    return zip_path


def folder_size_mb(folder):
    return round(sum(p.stat().st_size for p in Path(folder).rglob("*") if p.is_file()) / 1024**2, 2)

import torch

MEMBER = "TripleNeurons"
NOTEBOOK_ID = "NB4"
GROUP_NAME = "TripleNeurons"
HANDOFF_DIR = Path("/content/nb2_handoff")
OUT_DIR = Path("/content/nb4_outputs")
for folder in (HANDOFF_DIR, OUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LLM_DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
print("Device:", DEVICE)
if DEVICE == "cpu":
    print("WARNING: no GPU. Use Runtime > Change runtime type > T4 GPU, or each answer will take about a minute.")


Device: cuda


In [3]:
import platform, sys
import torch


def environment_rows():
    import importlib.metadata as md
    rows = {"Python": sys.version.split()[0], "Platform": platform.platform(),
            "GPU": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
            "CUDA available": torch.cuda.is_available()}
    for pkg in ["torch", "transformers", "sentence-transformers", "langchain-core",
                "langchain-text-splitters", "langchain-chroma", "langchain-huggingface",
                "chromadb", "pypdf", "pytesseract", "pdf2image", "gradio"]:
        try:
            rows[pkg] = md.version(pkg)
        except Exception:
            rows[pkg] = "not installed"
    return rows


for key, value in environment_rows().items():
    log("Environment", item=key, value=str(value))
    print(f"{key:<26}: {value}")

Python                    : 3.13.16
Platform                  : Linux-6.6.159+-x86_64-with-glibc2.39
GPU                       : Tesla T4
CUDA available            : True
torch                     : 2.14.1+cu130
transformers              : 5.22.2
sentence-transformers     : 6.2.0
langchain-core            : 1.7.4
langchain-text-splitters  : 1.1.3
langchain-chroma          : 1.1.0
langchain-huggingface     : 1.2.3
chromadb                  : 1.5.9
pypdf                     : not installed
pytesseract               : not installed
pdf2image                 : not installed
gradio                    : 6.31.0


## 3. Load the Notebook 2 hand-off (index, configuration, rag_core.py)

In [4]:
from google.colab import files
import sys

if not (HANDOFF_DIR / "config.json").exists():
    print("Upload TripleNeurons_NB2_Outputs.zip (the hand-off ZIP downloaded at the end of Notebook 2).")
    uploaded = files.upload()
    for filename in uploaded:
        if filename.lower().endswith(".zip"):
            with zipfile.ZipFile(filename) as zf:
                zf.extractall(HANDOFF_DIR)
            print("Extracted:", filename)

CONFIG = json.loads((HANDOFF_DIR / "config.json").read_text())
sys.path.insert(0, str(HANDOFF_DIR))
print(json.dumps({k: CONFIG[k] for k in ("chunk_size", "chunk_overlap", "top_k", "embedding_model", "llm")}, indent=1))
log("Handoff", item="NB2 configuration loaded", value=json.dumps(CONFIG.get("selected_config", {})))

Upload TripleNeurons_NB2_Outputs.zip (the hand-off ZIP downloaded at the end of Notebook 2).


Saving TripleNeurons_NB2_Outputs.zip to TripleNeurons_NB2_Outputs.zip
Extracted: TripleNeurons_NB2_Outputs.zip
{
 "chunk_size": 800,
 "chunk_overlap": 0,
 "top_k": 3,
 "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
 "llm": "Qwen/Qwen2.5-1.5B-Instruct"
}


## 4. Load the embedding model, the saved Chroma index and Qwen2.5-1.5B-Instruct

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from transformers import AutoTokenizer, AutoModelForCausalLM

embeddings = HuggingFaceEmbeddings(model_name=CONFIG["embedding_model"], model_kwargs={"device": DEVICE},
                                   encode_kwargs={"normalize_embeddings": True})
vectorstore = Chroma(collection_name=CONFIG["collection"], embedding_function=embeddings,
                     persist_directory=str(HANDOFF_DIR / "chroma_db"))
print("Vectors in saved database:", vectorstore._collection.count())

tokenizer = AutoTokenizer.from_pretrained(CONFIG["llm"])
try:
    model = AutoModelForCausalLM.from_pretrained(CONFIG["llm"], device_map={"": DEVICE}, dtype=LLM_DTYPE)
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(CONFIG["llm"], device_map={"": DEVICE}, torch_dtype=LLM_DTYPE)
model.eval()

import rag_core
from rag_core import WasteRAG
rag = WasteRAG(vectorstore, embeddings, tokenizer, model, top_k=CONFIG["top_k"],
               max_new_tokens=CONFIG["max_new_tokens"])
print("RAG core loaded from", rag_core.__file__)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (14 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

RAG core loaded from /content/nb2_handoff/rag_core.py


## 5. Quick check before launching
One known question, to confirm that the loaded pipeline answers with sources.

In [6]:
question = "How should I store kitchen waste while waiting for collection?"
result = rag.ask(question)
print("Q:", question)
print("A:", result["answer"])
for s in result["sources"]:
    print(f"   - {s['document']}, page {s['page']} (similarity {s['score']})")
print(f"Total time: {result['total_s']:.2f} s")
log("Interface_Check", question=question, answer=result["answer"],
    sources="; ".join(f"{s['document']} p.{s['page']}" for s in result["sources"]),
    total_s=round(result["total_s"], 3))

Q: How should I store kitchen waste while waiting for collection?
A: Place kitchen waste in a covered container. A can or a covered pail will work well. Never place in a plastic bag.
   - NSWMC Guidebook, page 22 (similarity 0.6716)
   - RA 9003 Information Material, page 1 (similarity 0.6256)
   - NSWMC Guidebook, page 35 (similarity 0.5488)
Total time: 2.25 s


## 6. Launch the interface
The cell prints a public `gradio.live` link (it works while this Colab runtime is running). Ask your questions there and take screenshots for the report.

In [7]:
import html
import re
import gradio as gr

GRADIO_MAJOR = int(gr.__version__.split(".")[0])

KNOWLEDGE_BASE = [  # label shown to the user, pages in the PDF
    ("RA 9003 (Ecological Solid Waste Management Act)", 27),
    ("RA 6969 (Toxic Substances and Hazardous Wastes Act)", 6),
    ("DENR DAO 2013-22 (hazardous wastes)", 71),
    ("NSWMC Guidebook", 56),
    ("RA 9003 Information Material", 1),
]
EXAMPLES = [
    "How should I store kitchen waste while waiting for collection?",
    "Are used diapers and Styrofoam food packs biodegradable, recyclable or residual waste?",
    "I have an old car battery at home. How should I classify and dispose of it?",
]
PASSAGE_CHARS = 700

CSS = """
.gradio-container {max-width: 1280px !important; background: #F6F7F4 !important;}
body, .gradio-container {font-family: Inter, system-ui, sans-serif;}
#tn-header {display:flex; align-items:center; gap:14px; padding:16px 20px; background:#FFFFFF;
  border:1px solid #DDE3DE; border-radius:16px;}
#tn-header .mark {width:44px; height:44px; border-radius:12px; background:#1F6B4A; display:flex;
  align-items:center; justify-content:center; flex:0 0 auto;}
#tn-header h1 {margin:0; font-size:20px; font-weight:600; color:#17201B;}
#tn-header p {margin:2px 0 0; font-size:13px; color:#4B5A52;}
.tn-card {background:#FFFFFF; border:1px solid #DDE3DE; border-radius:14px; padding:16px; color:#17201B;}
.tn-card h2 {margin:0 0 10px; font-size:15px; font-weight:600; color:#17201B;}
.tn-kb {list-style:none; margin:0; padding:0; font-size:13px;}
.tn-kb li {display:flex; justify-content:space-between; gap:8px; padding:5px 0; border-bottom:1px solid #EEF1EE;}
.tn-kb li:last-child {border-bottom:0;}
.tn-kb span.pp {color:#4B5A52; white-space:nowrap;}
.tn-note {background:#FFF5E1; border:1px solid #F0D9A8; border-radius:14px; padding:14px; font-size:13px;
  line-height:1.5; color:#5C3800;}
.tn-example button, button.tn-example {text-align:left !important; justify-content:flex-start !important;
  font-weight:400 !important; min-height:44px;}
#tn-chat {background:#FFFFFF; border:1px solid #DDE3DE; border-radius:16px; padding:20px; min-height:460px;
  max-height:640px; overflow-y:auto; color:#17201B;}
.tn-empty {color:#4B5A52; font-size:14px; text-align:center; padding:120px 20px;}
.tn-user {display:flex; justify-content:flex-end; margin:0 0 14px;}
.tn-user p {margin:0; max-width:78%; background:#1F6B4A; color:#FFFFFF; padding:12px 16px;
  border-radius:16px 16px 4px 16px; font-size:15px; line-height:1.5;}
.tn-bot {margin:0 0 20px; display:flex; flex-direction:column; gap:10px;}
.tn-answer {font-size:15px; line-height:1.6; color:#17201B; margin:0;}
.tn-refusal {background:#FFF5E1; border:1px solid #F0D9A8; border-radius:12px; padding:12px 14px;}
.tn-refusal .tn-answer {color:#3D2600; font-weight:500;}
.tn-refusal small {display:block; margin-top:6px; color:#5C3800; font-size:13px; line-height:1.5;}
.tn-sources {display:flex; flex-direction:column; gap:6px;}
.tn-sources details {border:1px solid #DDE3DE; border-radius:10px; background:#FFFFFF;}
.tn-sources details[open] {background:#F6F7F4;}
.tn-sources summary {cursor:pointer; padding:10px 12px; font-size:13px; color:#17201B; list-style-position:inside;}
.tn-sources summary b {font-weight:600;}
.tn-sources summary span {color:#4B5A52;}
.tn-sources blockquote {margin:0; padding:0 14px 12px; font-size:13px; line-height:1.55; color:#26302A;
  }
.tn-meta {font-size:12px; color:#4B5A52;}
#tn-send {background:#1F6B4A !important; color:#FFFFFF !important; border:0 !important;}
#tn-send:hover {background:#164D36 !important;}
@media (max-width: 760px) { #tn-side {order: 2;} #tn-main {order: 1;} }
.tn-thinking {color:#4B5A52; font-size:14px; font-style:italic;}
"""

HEADER_HTML = """
<div id="tn-header">
  <div class="mark" aria-hidden="true">
    <svg width="24" height="24" viewBox="0 0 24 24" fill="none" stroke="#FFFFFF" stroke-width="1.8"
      stroke-linecap="round" stroke-linejoin="round"><path d="M4 7h16"/><path d="M9 7V4h6v3"/>
      <path d="M6 7l1 13h10l1-13"/><path d="M10 11v6"/><path d="M14 11v6"/></svg>
  </div>
  <div><h1>Waste Segregation Assistant</h1>
  <p>Answers from 5 Philippine sources, with the document and page for every answer &middot; TripleNeurons</p></div>
</div>
"""

KB_HTML = ('<div class="tn-card"><h2>Knowledge base</h2><ul class="tn-kb">'
           + "".join(f"<li><span>{html.escape(n)}</span><span class='pp'>{p} pp.</span></li>"
                     for n, p in KNOWLEDGE_BASE)
           + "</ul></div>")

NOTE_HTML = ('<div class="tn-note">Information only, not legal advice. Collection schedules and local rules '
             'differ per barangay and city: ask your LGU or city environment office.</div>')


def render_turn(turn):
    """One question and its answer as HTML (all text from the user or the model is escaped)."""
    q = html.escape(turn["question"])
    out = [f'<div class="tn-user"><p>{q}</p></div>']
    if turn.get("pending"):
        out.append('<div class="tn-bot"><p class="tn-thinking">Searching the documents and writing an answer...</p></div>')
        return "".join(out)
    answer = html.escape(turn["answer"]).replace("\n", "<br>")
    if turn["refusal"]:
        body = (f'<div class="tn-refusal"><p class="tn-answer">{answer}</p>'
                '<small>This assistant only covers RA 9003, RA 6969, DAO 2013-22 and two NSWMC materials. '
                'For other laws or local rules, check the official source or your LGU.</small></div>')
    else:
        body = f'<p class="tn-answer">{answer}</p>'
    chips = []
    for s in turn["sources"]:
        chips.append(
            f'<details><summary><b>{html.escape(str(s["document"]))}</b> '
            f'<span>&middot; page {s["page"]} &middot; similarity {s["score"]:.2f}</span></summary>'
            f'<blockquote>{html.escape(s["passage"])}</blockquote></details>')
    label = "Retrieved passages (not used: answer refused)" if turn["refusal"] else "Sources (click to read the passage)"
    out.append(f'<div class="tn-bot">{body}<div class="tn-meta">{label}</div>'
               f'<div class="tn-sources">{"".join(chips)}</div>'
               f'<div class="tn-meta">Answered in {turn["seconds"]:.2f} s</div></div>')
    return "".join(out)


def render_chat(history):
    if not history:
        return ('<div id="tn-chat"><div class="tn-empty">Ask a question about classifying or segregating waste, '
                'or pick an example on the left.</div></div>')
    return '<div id="tn-chat">' + "".join(render_turn(t) for t in history) + "</div>"


def answer_question(question, history):
    question = (question or "").strip()
    if not question:
        yield gr.update(), history, render_chat(history)
        return
    history = list(history) + [{"question": question, "pending": True}]
    yield "", history, render_chat(history)
    result = rag.ask(question)
    passages = []
    for d in result["docs"]:
        text = re.sub(r"\s+", " ", d.page_content).strip()      # OCR line breaks -> spaces
        passages.append(text[:PASSAGE_CHARS] + ("..." if len(text) > PASSAGE_CHARS else ""))
    sources = [{**s, "passage": p} for s, p in zip(result["sources"], passages)]
    history[-1] = {"question": question, "answer": result["answer"], "refusal": result["refusal"],
                   "sources": sources, "seconds": result["total_s"]}
    log("App_Usage", question=question, answer=result["answer"], refusal_flag=result["refusal"],
        sources="; ".join(f"{s['document']} p.{s['page']} ({s['score']})" for s in result["sources"]),
        retrieval_s=round(result["retrieval_s"], 3), generation_s=round(result["generation_s"], 3),
        total_s=round(result["total_s"], 3), new_tokens=result["new_tokens"])
    yield "", history, render_chat(history)


def new_chat():
    return [], render_chat([]), ""


FORCE_LIGHT = "() => { document.body.classList.remove('dark'); }"
blocks_kwargs = {"title": "Waste Segregation Assistant"}
style_kwargs = {"css": CSS, "theme": gr.themes.Base(primary_hue="green", font=[gr.themes.GoogleFont("Inter"), gr.themes.Font("system-ui"), gr.themes.Font("sans-serif")]),
                "js": FORCE_LIGHT}
if GRADIO_MAJOR < 6:            # Gradio 6 moved css/theme/js from Blocks() to launch()
    blocks_kwargs.update(style_kwargs)

with gr.Blocks(**blocks_kwargs) as demo:
    history_state = gr.State([])
    gr.HTML(HEADER_HTML)
    with gr.Row(equal_height=False):
        with gr.Column(scale=1, min_width=260, elem_id="tn-side"):
            gr.HTML(KB_HTML)
            gr.Markdown("**Try asking**")
            example_buttons = [gr.Button(e, size="sm", variant="secondary", elem_classes=["tn-example"])
                               for e in EXAMPLES]
            gr.HTML(NOTE_HTML)
        with gr.Column(scale=3, min_width=360, elem_id="tn-main"):
            chat_html = gr.HTML(render_chat([]))
            with gr.Row(equal_height=True):
                question_box = gr.Textbox(label="Ask about waste classification or segregation",
                                          placeholder="e.g. Where do busted fluorescent lamps go?",
                                          scale=5, autofocus=True)
                send_button = gr.Button("Send", variant="primary", scale=1, min_width=100, elem_id="tn-send")
            with gr.Row():
                new_button = gr.Button("New chat", variant="secondary", size="sm")
            gr.Markdown("Every answer lists the document and page it came from. Open a source to check it before acting.")

    outputs = [question_box, history_state, chat_html]
    SCROLL = "() => { const c = document.getElementById('tn-chat'); if (c) c.scrollTop = c.scrollHeight; }"
    send_button.click(answer_question, [question_box, history_state], outputs).then(None, None, None, js=SCROLL)
    question_box.submit(answer_question, [question_box, history_state], outputs).then(None, None, None, js=SCROLL)
    def example_handler(text):
        def handler(history):
            yield from answer_question(text, history)
        return handler

    for button, text in zip(example_buttons, EXAMPLES):
        button.click(example_handler(text), [history_state], outputs).then(None, None, None, js=SCROLL)
    new_button.click(new_chat, None, [history_state, chat_html, question_box])

launch_kwargs = {"share": True, "debug": False}
if GRADIO_MAJOR >= 6:
    launch_kwargs.update(style_kwargs)


demo.launch(**launch_kwargs)
print("Open the public gradio.live link above. Every question asked in the interface is added to the App_Usage log.")


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d16d3e7a753feb3082.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Open the public gradio.live link above. Every question asked in the interface is added to the App_Usage log.


## 7. Save the usage log
Run this after you finish asking questions in the interface. It downloads `TripleNeurons_NB4_Interface_Logs.zip` (environment, hand-off check and App_Usage sheets) for Deliverable 4.

In [9]:
import pandas as pd

save_logs(OUT_DIR / "logs")
if "App_Usage" in LOGS:
    usage = pd.DataFrame(LOGS["App_Usage"])
    print(f"{len(usage)} questions were asked in the interface.")
    display(usage[["timestamp", "question", "refusal_flag", "sources", "total_s"]])
else:
    print("No questions were asked in the interface yet.")

zip_path = zip_folder(OUT_DIR, "/content/TripleNeurons_NB4_Interface_Logs.zip")
files.download(zip_path)

Saved log sheets: ['App_Usage', 'Environment', 'Handoff', 'Interface_Check']
5 questions were asked in the interface.


,timestamp,question,refusal_flag,sources,total_s
0,2026-11-10T22:40:39+08:00,Where should I throw busted fluorescent lamps?,False,NSWMC Guidebook p.44 (0.3665); RA 9003 Informa...,3.651
1,2026-11-10T22:40:51+08:00,Are used diapers and Styrofoam food packs biod...,False,NSWMC Guidebook p.12 (0.63); NSWMC Guidebook p...,2.032
2,2026-11-10T22:40:58+08:00,I have an old car battery at home. How should ...,False,NSWMC Guidebook p.19 (0.5203); DAO 2013-22 p.6...,2.463
3,2026-11-10T22:41:04+08:00,What is the penalty for littering or dumping w...,False,NSWMC Guidebook p.16 (0.6805); NSWMC Guidebook...,3.660
4,2026-11-10T22:41:06+08:00,What does the Extended Producer Responsibility...,True,RA 9003 p.17 (0.6246); RA 9003 p.17 (0.5877); ...,0.686


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>